### Load imports and secrets

In [19]:
import os
import sys
from dotenv import load_dotenv

from openai import OpenAI
from anthropic import Anthropic

load_dotenv()

True

In [15]:
print("OPENAI Ключ найден:" if "OPENAI_API_KEY" in os.environ else "Ключ не найден")
print("ANTHROPIC Ключ найден:" if "ANTHROPIC_API_KEY" in os.environ else "Ключ не найден")
print("GOOGLE Ключ найден:" if "GOOGLE_API_KEY" in os.environ else "Ключ не найден")

OPENAI Ключ найден:
ANTHROPIC Ключ найден:
GOOGLE Ключ найден:


## 1. OpenAI Chat Completions

In [12]:
def ask_with_context(context, question):
    client = OpenAI()

    messages = [
        {
            "role": "system",
            "content": "Answer based only on the provided context."
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion:\n{question}"
        },
    ]

    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=messages
    )

    return response.choices[0].message.content

In [13]:
# Usage
context = "RAG stands for Retrieval-Augmented Generation."
question = "What does RAG stand for?"
answer = ask_with_context(context, question)
print(answer)

Retrieval-Augmented Generation.


## 2. OpenAI Whisper Speech-to-Text

In [14]:
client = OpenAI()

audio_path = "../../datasets/audio_files/harvard.wav"

with open(audio_path, "rb") as audio_file:
    transcript = client.audio.transcriptions.create(
        model="gpt-4o-mini-transcribe",
        file=audio_file,
    )

print(transcript.text)

The stale smell of old beer lingers. It takes heat to bring out the odor. A cold dip restores health and zest. A salt pickle tastes fine with ham. Tacos al pastor are my favorite. A zestful food is the hot cross bun.


## 3. Gemini API Example using the OpenAI SDK

In [16]:
google_client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

google_client.models.list()

SyncPage[Model](data=[Model(id='models/gemini-2.5-flash', created=None, object='model', owned_by='google', display_name='Gemini 2.5 Flash'), Model(id='models/gemini-2.5-pro', created=None, object='model', owned_by='google', display_name='Gemini 2.5 Pro'), Model(id='models/gemini-2.0-flash', created=None, object='model', owned_by='google', display_name='Gemini 2.0 Flash'), Model(id='models/gemini-2.0-flash-001', created=None, object='model', owned_by='google', display_name='Gemini 2.0 Flash 001'), Model(id='models/gemini-2.0-flash-lite-001', created=None, object='model', owned_by='google', display_name='Gemini 2.0 Flash-Lite 001'), Model(id='models/gemini-2.0-flash-lite', created=None, object='model', owned_by='google', display_name='Gemini 2.0 Flash-Lite'), Model(id='models/gemini-2.5-flash-preview-tts', created=None, object='model', owned_by='google', display_name='Gemini 2.5 Flash Preview TTS'), Model(id='models/gemini-2.5-pro-preview-tts', created=None, object='model', owned_by='goo

In [17]:
resp = google_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ],
)

print(resp.choices[0].message.content)

The capital of France is **Paris**.


## 4. Anthropic Claude Example

In [20]:
antropic_client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

In [22]:
response = antropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=200,
    messages=[
        {
            "role": "user",
            "content": "Explain how vector databases work in "
                       "simple terms.",
        }
    ],
)

print(response.content[0].text)

# How Vector Databases Work (Simple Explanation)

## The Basic Idea

Instead of storing traditional data (text, numbers, dates), vector databases store **numerical representations of meaning**.

Think of it like this:
- **Regular database**: Stores the word "dog"
- **Vector database**: Stores that "dog" means [0.2, 0.8, 0.1, ...] (hundreds of numbers)

## The Process

### 1. **Converting to Vectors**
Data gets converted into vectors using AI models:
- Text → numbers that capture meaning
- Images → numbers that capture visual features
- The vector represents "what something is about"

### 2. **Storing Vectors**
The database saves these numerical lists, indexed for fast searching.

### 3. **Finding Similar Items**
Instead of exact matches, you find similar vectors:
```
Query: "puppy


## 5. Deploy local LLMs using Ollama

In [23]:
print("Pulling models...")
# !ollama pull qwen3:4b
# !ollama pull llama3.2
# !ollama pull gemma3:2b

print("Models pulled successfully. You can now re-run the Ollama examples.")

Pulling models...
Models pulled successfully. You can now re-run the Ollama examples.


❯ ollama pull qwen3:4b\
pulling manifest\
pulling 3e4cb1417446: 100% ▕██████████████████████████████████████████████▏ 2.5 GB\
pulling 2d54db2b9bb2: 100% ▕██████████████████████████████████████████████▏ 1.5 KB\
pulling d18a5cc71b84: 100% ▕██████████████████████████████████████████████▏  11 KB\
pulling cff3f395ef37: 100% ▕██████████████████████████████████████████████▏  120 B\
pulling e18a783aae55: 100% ▕██████████████████████████████████████████████▏  487 B\
verifying sha256 digest\
writing manifest\
success\

In [ ]:
# Point the client to your local Ollama server
ollama_client = OpenAI(
    base_url="http://host.docker.internal:11434/v1",
    api_key="ollama",  # Ollama does not require a real key, but the SDK expects one
)
# Ollama running on Windows host
# Jupyter running inside WSL
# host.docker.internal bridges WSL/container

In [39]:
response = ollama_client.chat.completions.create(
    model="qwen3:4b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": "What is retrieval augmented generation?"
        },
    ],
)

print(response.choices[0].message.content)

**Retrieval Augmented Generation (RAG)** is a technique used in **natural language processing (NLP)** and **large language models (LLMs)** to improve the accuracy, relevance, and timeliness of generated text by **combining external knowledge** with the model's capabilities. Here's a clear breakdown:

---

### 🔍 **How RAG Works (Simple Explanation)**
1. **Retrieval**:  
   The system first retrieves **relevant information** from a knowledge base (e.g., documents, databases, websites) based on the user's query.  
   *Example*: If you ask, *"What's the latest climate change report from the UN?"*, the system fetches the most recent UN climate reports from a trusted source.

2. **Generation**:  
   The retrieved information is then used to **generate a precise, context-aware response** using an LLM (like GPT).  
   *Example*: The LLM uses the retrieved UN report to craft an answer like: *"The UN's latest climate report (2024) highlights a 1.5°C temperature rise... [details]."*.

---

### 💡 

## 6. Pydantic Structured Output

In [40]:
from openai import OpenAI
from pydantic import BaseModel
from datetime import date
from typing import List

class LineItem(BaseModel):
    description: str
    quantity: int
    total: float

class Invoice(BaseModel):
    invoice_number: str
    invoice_date: date
    supplier: str
    items: List[LineItem]
    total_due: float

client = OpenAI()

completion = client.chat.completions.parse(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": "Extract the invoice data from the provided context."},
        {"role": "user", "content": "Invoice #12345, Date: 2024-01-15, Supplier: Tech Corp. Item: Laptop, Qty: 2, Total: $2000. Item: Mouse, Qty: 5, Total: $100. Total Due: $2100"}
    ],
    response_format=Invoice,
)

invoice = completion.choices[0].message.parsed
print(invoice)

invoice_number='12345' invoice_date=datetime.date(2024, 1, 15) supplier='Tech Corp' items=[LineItem(description='Laptop', quantity=2, total=2000.0), LineItem(description='Mouse', quantity=5, total=100.0)] total_due=2100.0
